In [7]:
import pandas as pd
from pymongo import MongoClient
import os

In [8]:
os.chdir(r'\Users\DAY\Desktop\DIPLOMADO_ANALISIS_DE_DATOS')
df = pd.read_csv("Analista de Datos M49 Tweets (1).csv")
df.head()

,tweet_id,airline_sentiment,airline_sentiment_confidence,negativereason,negativereason_confidence,airline,airline_sentiment_gold,name,negativereason_gold,retweet_count,text,tweet_coord,tweet_created,tweet_location,user_timezone
0,570306133677760513,neutral,1.0000,NaN,NaN,Virgin America,NaN,cairdin,NaN,0,@VirginAmerica What @dhepburn said.,NaN,2015-02-24 11:35:52 -0800,NaN,Eastern Time (US & Canada)
1,570301130888122368,positive,0.3486,NaN,0.0000,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica plus you've added commercials t...,NaN,2015-02-24 11:15:59 -0800,NaN,Pacific Time (US & Canada)
2,570301083672813571,neutral,0.6837,NaN,NaN,Virgin America,NaN,yvonnalynn,NaN,0,@VirginAmerica I didn't today... Must mean I n...,NaN,2015-02-24 11:15:48 -0800,Lets Play,Central Time (US & Canada)
3,570301031407624196,negative,1.0000,Bad Flight,0.7033,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica it's really aggressive to blast...,NaN,2015-02-24 11:15:36 -0800,NaN,Pacific Time (US & Canada)
4,570300817074462722,negative,1.0000,Can't Tell,1.0000,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica and it's a really big bad thing...,NaN,2015-02-24 11:14:45 -0800,NaN,Pacific Time (US & Canada)


In [9]:
df.shape

(14640, 15)

In [10]:
df.columns

Index(['tweet_id', 'airline_sentiment', 'airline_sentiment_confidence',
       'negativereason', 'negativereason_confidence', 'airline',
       'airline_sentiment_gold', 'name', 'negativereason_gold',
       'retweet_count', 'text', 'tweet_coord', 'tweet_created',
       'tweet_location', 'user_timezone'],
      dtype='object')

In [11]:
df.info

<bound method DataFrame.info of                  tweet_id airline_sentiment  airline_sentiment_confidence  \
0      570306133677760513           neutral                        1.0000   
1      570301130888122368          positive                        0.3486   
2      570301083672813571           neutral                        0.6837   
3      570301031407624196          negative                        1.0000   
4      570300817074462722          negative                        1.0000   
...                   ...               ...                           ...   
14635  569587686496825344          positive                        0.3487   
14636  569587371693355008          negative                        1.0000   
14637  569587242672398336           neutral                        1.0000   
14638  569587188687634433          negative                        1.0000   
14639  569587140490866689           neutral                        0.6771   

               negativereason  negativereas

In [12]:
#Palbras clave
keywords = ["delay", "cancelled", "service", "flight", "thanks"]

In [13]:
#Cuantos tweets contienen cada palabra
for word in keywords:
    count = df["text"].str.contains(word, case=False, na=False).sum()
    print(f"{word}: {count}")

delay: 960
cancelled: 1020
service: 1116
flight: 4287
thanks: 1078


In [14]:
for word in keywords:
    print(f"\nEjemplos para: {word}")
    print(df[df["text"].str.contains(word, case=False, na=False)][["airline", "airline_sentiment", "text"]].head(3))


Ejemplos para: delay
            airline airline_sentiment  \
82   Virgin America          negative   
115  Virgin America          negative   
119  Virgin America          positive   

                                                  text  
82   @VirginAmerica you're the best!! Whenever I (b...  
115  @VirginAmerica should I be concerned that I am...  
119  @VirginAmerica Love the team running Gate E9 a...  

Ejemplos para: cancelled
            airline airline_sentiment  \
83   Virgin America          negative   
92   Virgin America          negative   
129  Virgin America           neutral   

                                                  text  
83   @VirginAmerica I have no interesting flying wi...  
92   @VirginAmerica I like the TV and interesting v...  
129  @VirginAmerica is flight 882 Cancelled Flightl...  

Ejemplos para: service
           airline airline_sentiment  \
33  Virgin America          negative   
71  Virgin America           neutral   
78  Virgin America    

In [15]:
from pymongo import MongoClient
import pandas as pd

# conexión
client = MongoClient("mongodb://localhost:27017/")

# base de datos
db = client["tweets_db"]

# colección
collection = db["tweets"]

print("Conexión exitosa")

Conexión exitosa


In [16]:
df_1000 = df.head(1000).copy()

records = df_1000.to_dict(orient="records")

collection.insert_many(records)

print("Tweets insertados:", len(records))

Tweets insertados: 1000


In [17]:
collection.count_documents({})

1000

In [18]:
#Filtros de búsqueda
#Filtro 1
filter_1 = {"airline": "United","airline_sentiment": "negative"}

results_1 = list(collection.find(filter_1,{"_id": 0, "airline": 1, "airline_sentiment": 1, "text": 1}).limit(5))

results_1

[{'airline_sentiment': 'negative',
  'airline': 'United',
  'text': "@united still no refund or word via DM. Please resolve this issue as your Cancelled Flightled flight was useless to my assistant's trip."},
 {'airline_sentiment': 'negative',
  'airline': 'United',
  'text': "@united Delayed due to lack of crew and now delayed again because there's a long line for deicing... Still need to improve service #united"},
 {'airline_sentiment': 'negative',
  'airline': 'United',
  'text': '@united Your ERI-ORD express connections are hugely popular .. now if only we could have an ERI-EWR hop! :)'},
 {'airline_sentiment': 'negative',
  'airline': 'United',
  'text': '@united you think you boarded flight AU1066 too early? I think so.'},
 {'airline_sentiment': 'negative',
  'airline': 'United',
  'text': '@united Gate agent hooked me up with alternate flights. If you have a way to PREVENT the constant issues, that would rock.'}]

In [19]:
#Filtro 2
#Por palabra clave
filter_2 = {
    "text": {"$regex": "delay", "$options": "i"}}

results_2 = list(collection.find(filter_2,{"_id": 0, "airline": 1, "airline_sentiment": 1, "text": 1}).limit(5))

results_2

[{'airline_sentiment': 'negative',
  'airline': 'Virgin America',
  'text': "@VirginAmerica you're the best!! Whenever I (begrudgingly) use any other airline I'm delayed and Late Flight :("},
 {'airline_sentiment': 'negative',
  'airline': 'Virgin America',
  'text': '@VirginAmerica should I be concerned that I am about to fly on a plane that needs to be delayed due to a "tech stop"?'},
 {'airline_sentiment': 'positive',
  'airline': 'Virgin America',
  'text': '@VirginAmerica Love the team running Gate E9 at LAS tonight. Waited for a delayed flight, and they kept things entertaining'},
 {'airline_sentiment': 'negative',
  'airline': 'Virgin America',
  'text': '@VirginAmerica I like the customer service but a 40 min delay just for connecting passengers seems too long. VA370'},
 {'airline_sentiment': 'negative',
  'airline': 'Virgin America',
  'text': '@VirginAmerica Another delayed flight? #likingyoulessandless'}]